In [69]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch import nn
import math
import torch.optim as optim
import spacy
from torchtext.data import Field,BucketIterator
from torchtext.datasets import Multi30k
from torch.nn.utils.rnn import pad_sequence
import nltk
from nltk.translate.bleu_score import sentence_bleu
nn.Transformer()

Transformer(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, o

In [70]:
class ScaleDotProductAttention(nn.Module):
    def __init__(self):
        super(ScaleDotProductAttention,self).__init__()
        self.softmax=nn.Softmax(dim=-1)
    def forward(self,q,k,v,mask=None,e=1e-12):
        d_tensor=k.shape[-1]
        k_T=k.transpose(2,3)
        score=(q@k_T)/math.sqrt(d_tensor)
        if mask is not None:
            score=score.masked_fill(mask==0,-10000)
        score=self.softmax(score)
        v=score@v
        return v

In [71]:
class MultiHeadAttention(nn.Module):
    def __init__(self,num_heads,d_model):
        super(MultiHeadAttention,self).__init__()
        self.w_k=nn.Linear(d_model,d_model)
        self.w_q=nn.Linear(d_model,d_model)
        self.w_v=nn.Linear(d_model,d_model)
        self.w_o=nn.Linear(d_model,d_model)
        self.num_heads=num_heads
        self.attention=ScaleDotProductAttention()

    def changeshape(self,tensor):#改变形状，得到多头
        batch_size,vocab_legth,d_model=tensor.size()
        d_head=d_model/self.num_heads
        tensor=tensor.reshape(batch_size,vocab_legth,self.num_heads,-1)
        tensor=tensor.permute(0,2,1,3)
        return tensor
    def retainshape(self,tensor):#最后的结果恢复形状
        batch_size,num_heads,vocab_length,d_head=tensor.size()
        d_model=num_heads*d_head
        tensor=tensor.permute(0,2,1,3)
        tensor=tensor.reshape(batch_size,vocab_length,-1)
        return tensor
    def forward(self,q,k,v,mask=None):
        q=self.w_q(q)
        k=self.w_k(k)
        v=self.w_v(v)
        #多头操作
        q=self.changeshape(q)
        k=self.changeshape(k)
        v=self.changeshape(v)
        result=self.attention(q,k,v,mask)
        result=self.retainshape(result)
        result=self.w_o(result)
        return result



In [72]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self,d_model,ffn_num_hiddens,dropout):#dropout
        super(PositionwiseFeedForward,self).__init__()
        self.linear1=nn.Linear(d_model,ffn_num_hiddens)
        self.linear2=nn.Linear(ffn_num_hiddens,d_model)
        self.relu=nn.ReLU()
        self.dropout=nn.Dropout(dropout)
    def forward(self,X):
        Y=self.relu(self.linear1(X))
        return self.linear2(self.dropout(Y))

In [73]:
class PositionwiseFeedForward_conv(nn.Module):
    def __init__(self,d_model,ffn_num_hiddens):
        super(PositionwiseFeedForward_conv,self).__init__()
        self.conv1=nn.Conv2d(d_model,ffn_num_hiddens,kernel_size=1)
        self.conv2=nn.Conv2d(ffn_num_hiddens,d_model,kernel_size=1)
    def forward(self,X):
        Y=self.conv1(X)
        return self.conv2(Y)


In [74]:
class encoder(nn.Module):
    def __init__(self,d_model,num_heads,ffn_num_hiddens,dropout):#dropout
        super(encoder,self).__init__()
        self.num_heads=num_heads
        self.ffn_num_hiddens=ffn_num_hiddens
        self.attention=MultiHeadAttention(num_heads=num_heads,d_model=d_model)
        self.ffn=PositionwiseFeedForward(d_model,ffn_num_hiddens,dropout)
        self.layernorm1=nn.LayerNorm(d_model)
        self.layernorm2=nn.LayerNorm(d_model)
        self.dropout1=nn.Dropout(dropout)
        self.dropout2=nn.Dropout(dropout)


    def forward(self,X,mask=None):
        X_1=self.attention(q=X,k=X,v=X,mask=mask)
        X_1=self.layernorm1(self.dropout1(X_1)+X)
        Y=self.ffn(X_1)
        Y=self.layernorm2(self.dropout2(Y)+X_1)
        return Y



In [75]:
class PositonEncoding(nn.Module):
    def __init__(self,d_model,max_len,device):
        super(PositonEncoding,self).__init__()
        self.code=torch.zeros(max_len,d_model,device=device)
        position=torch.arange(0,max_len,device=device).unsqueeze(1).float()
        down=10000**(torch.arange(0,d_model,2,device=device).float()/d_model)
        self.code[:,0::2]=torch.sin(position/down)
        self.code[:,1::2]=torch.cos(position/down)

    def forward(self,X):
        batch_size,seq_len=X.shape[0],X.shape[1]
        return self.code[:seq_len,:]       


In [76]:
class TokenEmbedding(nn.Embedding):
    def __init__(self,vocab_size,d_model):
        super(TokenEmbedding,self).__init__(vocab_size,d_model,padding_idx=1)

In [77]:
class TransformerEmbedding(nn.Module):
    def __init__(self,vocab_size,d_model,max_len,device):#dropout
        super(TransformerEmbedding,self).__init__()
        self.embedding=TokenEmbedding(vocab_size,d_model)
        self.positionencoding=PositonEncoding(d_model,max_len,device=device)
    def forward(self,X):
        tok_emb=self.embedding(X)
        pos_emb=self.positionencoding(X)
        return tok_emb+pos_emb

In [78]:
class decoder(nn.Module):
    def __init__(self,num_heads,d_model,ffn_num_hiddens,dropout):#dropout
        super(decoder,self).__init__()
        self.attention1=MultiHeadAttention(num_heads,d_model)
        self.attention2=MultiHeadAttention(num_heads,d_model)
        self.ffn=PositionwiseFeedForward(d_model,ffn_num_hiddens,dropout)
        self.layernorm1=nn.LayerNorm(d_model)
        self.layernorm2=nn.LayerNorm(d_model)
        self.layernorm3=nn.LayerNorm(d_model)
        self.dropout1=nn.Dropout(dropout)
        self.dropout2=nn.Dropout(dropout)
        self.dropout3=nn.Dropout(dropout)


    def forward(self,encoder_output,X,src_mask,trg_mask):
        X_1=self.attention1(q=X,k=X,v=X,mask=trg_mask)
        X_1=self.layernorm1(X+self.dropout1(X_1))
        Y=self.attention2(q=X_1,k=encoder_output,v=encoder_output,mask=src_mask)
        Y=self.layernorm2(self.dropout2(Y)+X_1)
        Z=self.ffn(Y)
        Z=self.layernorm3(self.dropout3(Z)+Y)
        return Z

In [79]:
class TransformerEncoder(nn.Module):
    def __init__(self,vocab_size,d_model,max_len,num_heads,
                 ffn_num_hiddens,num_blocks,dropout,device):
        super(TransformerEncoder,self).__init__()
        self.embedding=TransformerEmbedding(vocab_size,d_model,max_len,device)
        self.layers=nn.ModuleList([encoder(d_model,num_heads,ffn_num_hiddens,dropout)
                                    for _ in range(num_blocks)])

    def forward(self,X,src_mask):
        X=self.embedding(X)
        for layer in self.layers:
            X=layer(X,src_mask)
        return X        

In [80]:
class TransformerDecoder(nn.Module):
    def __init__(self,dec_vocab_size,d_model,max_len,num_heads,
                 ffn_num_hiddens,num_blocks,dropout,device):
        super(TransformerDecoder,self).__init__()
        self.embedding=TransformerEmbedding(dec_vocab_size,d_model,max_len,device)
        self.layers=nn.ModuleList([decoder(num_heads,d_model,ffn_num_hiddens,dropout) 
                                   for _ in range(num_blocks) ])
        self.linear=nn.Linear(d_model,dec_vocab_size)
        # self.softmax=nn.Softmax(dim=-1)
    def forward(self,X,enc_output,src_mask,trg_mask):
        X=self.embedding(X)
        for layer in self.layers:
            X=layer(enc_output,X,src_mask,trg_mask)
        output=self.linear(X)
        return output
    

In [ ]:
# class Transformer(nn.Module):
#     def __init__(self,src_pad_idx,trg_pad_idx,trg_sos_idx,enc_vocab_size,dec_vocab_size,
#                  d_model,num_heads,max_len,ffn_num_hiddens,num_blocks,dropout,device):
#         super(Transformer,self).__init__()
#         self.src_pad_idx=src_pad_idx
#         self.trg_pad_idx=trg_pad_idx
#         self.trg_sos_idx=trg_sos_idx
#         self.device=device
#         self.encoder=TransformerEncoder(enc_vocab_size,d_model,max_len,num_heads,
#                                         ffn_num_hiddens,num_blocks,dropout,device)
#         self.decoder=TransformerDecoder(dec_vocab_size,d_model,max_len,num_heads,
#                                         ffn_num_hiddens,num_blocks,dropout,device)
#     def make_src_mask(self,X):
#         src_mask=(X!=self.src_pad_idx).unsqueeze(1).unsqueeze(2)
#         return src_mask
#     def make_trg_mask(self,Y):
#         Y_pad=(Y!=self.trg_pad_idx).unsqueeze(1).unsqueeze(2)
#         Y_len=Y.shape[1]
#         trg_mask=torch.tril(torch.ones(Y_len,Y_len,device=self.device)).bool()
#         return Y_pad & trg_mask
#     def forward(self,X,Y):
#         src_mask=self.make_src_mask(X)
#         trg_mask=self.make_trg_mask(Y)
#         enc_output=self.encoder(X,src_mask)
#         output=self.decoder(Y,enc_output,src_mask,trg_mask)
#         return output    
class Transformer(nn.Module):
    def __init__(self,src_pad_idx,trg_pad_idx,trg_sos_idx,enc_vocab_size,dec_vocab_size,
                 d_model,num_heads,max_len,ffn_num_hiddens,num_blocks,dropout,device):
        super(Transformer,self).__init__()
        self.src_pad_idx=src_pad_idx
        self.trg_pad_idx=trg_pad_idx
        self.trg_sos_idx=trg_sos_idx
        self.device=device
        self.encoder=TransformerEncoder(enc_vocab_size,d_model,max_len,num_heads,
                                        ffn_num_hiddens,num_blocks,dropout,device)
        self.decoder=TransformerDecoder(dec_vocab_size,d_model,max_len,num_heads,
                                        ffn_num_hiddens,num_blocks,dropout,device)
    def make_pad_mask(self,q,k,q_pad_idx,k_pad_idx):
        len_q,len_k=q.size(1),k.size(1)
        q=(q!=q_pad_idx).unsqueeze(1).unsqueeze(3)
        q=q.repeat(1,1,1,len_k)
        k=(k!=k_pad_idx).unsqueeze(1).unsqueeze(2)
        k=k.repeat(1,1,len_q,1)
        mask=k&q
        return mask
    def make_no_peak_mask(self,q,k):
        len_q,len_k=q.size(1),k.size(1)  
        mask=torch.tril(torch.ones(len_q,len_k,device=self.device)).bool()
        return mask
    def forward(self,src,trg):
        src_mask=self.make_pad_mask(src,src,self.src_pad_idx,self.src_pad_idx)
        src_trg_mask=self.make_pad_mask(trg,src,self.trg_pad_idx,self.src_pad_idx)
        trg_mask=self.make_pad_mask(trg,trg,self.trg_pad_idx,self.trg_pad_idx)*self.make_no_peak_mask(trg,trg)

        enc_output=self.encoder(src,src_mask)
        output=self.decoder(trg,enc_output,src_trg_mask,trg_mask)
        return output    

In [82]:
spacy_en=spacy.load('en_core_web_sm')
spacy_de=spacy.load('de_core_news_sm')
def tokenize_en(text):
    return [tok.text for tok in spacy_en.tokenizer(text)]
def tokenize_de(text):
    return [tok.text for tok in spacy_de.tokenizer(text)]

src=Field(tokenize=tokenize_en,init_token='<sos>',eos_token='<eos>',lower=True,batch_first=True)
trg=Field(tokenize=tokenize_de,init_token='<sos>',eos_token='<eos>',lower=True,batch_first=True)
#Field自动填充pad

# Download and load the Multi30k dataset
train_data, valid_data, test_data = Multi30k.splits(exts=('.en', '.de'), fields=(src, trg))

src.build_vocab(train_data,max_size=10000,min_freq=2)
trg.build_vocab(train_data,max_size=10000,min_freq=2)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

src_pad_idx=src.vocab.stoi['<pad>']
trg_pad_idx=trg.vocab.stoi['<pad>']
trg_sos_idx=trg.vocab.stoi['<sos>']

enc_vocab_size=len(src.vocab)
dec_vocab_size=len(trg.vocab)
batch_size=32
train_iter,valid_iter,test_iter=BucketIterator.splits((train_data,valid_data,test_data),
                                                      batch_size=batch_size,device=device)
#BucketIterator自动填充pad,保证每个小批量中的序列长度一致


In [83]:
src_pad_idx=src.vocab.stoi['<pad>']
trg_pad_idx=trg.vocab.stoi['<pad>']
trg_sos_idx=trg.vocab.stoi['<sos>']

enc_vocab_size=len(src.vocab)
dec_vocab_size=len(trg.vocab)

In [84]:
d_model=64
num_heads=8
num_blocks=2
ffn_num_hiddens=256


max_len=128
dropout=0.1
clip=1
model=Transformer(src_pad_idx,trg_pad_idx,trg_sos_idx,enc_vocab_size,dec_vocab_size,
                  d_model,num_heads,max_len,ffn_num_hiddens,num_blocks,dropout,device)
model=model.to(device)
def init_weights(m):
    if type(m)==nn.Linear:
        nn.init.xavier_uniform_(m.weight)
model.apply(init_weights)
optimizer=optim.Adam(model.parameters(),lr=0.0001,betas=(0.9,0.98),eps=1e-9)
criterion=nn.CrossEntropyLoss(ignore_index=trg_pad_idx)
scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer,factor=0.1,patience=10)

In [85]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The model has {count_parameters(model):,} tarinable parameters')

The model has 1,623,661 tarinable parameters


In [86]:
def train(model,iterator,optimizer,criterion,clip):
    model.train()
    epoch_loss=0
    for i,batch in enumerate(iterator):
        src=batch.src
        trg=batch.trg
        optimizer.zero_grad()
        output=model(src,trg[:,:-1])
        output_dim=output.shape[-1]
        output=output.contiguous().view(-1,output_dim)
        trg=trg[:,1:].contiguous().view(-1)
        loss=criterion(output,trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),clip)
        optimizer.step()
        epoch_loss+=loss.item()
    return epoch_loss/len(iterator)


In [87]:
def evaluate(model,data,criterion):
    model.eval()
    losses=[]
    for batch in data:
        X=batch.src
        Y=batch.trg
        Y_input=Y[:,:-1]
        Y_target=Y[:,1:]
        output=model(X,Y_input)
        loss=criterion(output,Y_target)
        losses.append(loss.item())
    return sum(losses)/len(losses)


In [88]:
def run_epoch(num_epochs):
    for epoch in range(num_epochs):
        train_loss=train(model,train_iter,optimizer,criterion,clip)
        # valid_loss=evaluate(model,valid_iter,criterion)
        # scheduler.step(valid_loss)
        print(f'epoch:{epoch+1},train_loss:{train_loss:.3f}')

原序列与目标序列的长度是不同的，那这里的问题是什么意思呢

In [89]:
# run_epoch(10)

In [90]:
def vision(epoch,y1,y2):
    fig,ax=plt.subplots()
    ax.set_xlim(1,len(epoch))
    ax.set_xticks(np.arange(1,len(epoch)+1,1))
    ax.plot(epoch,y1,label='train_loss')
    ax.plot(epoch,y2,label='valid_loss')
    # ax.plot(epoch,train_loss,label='train_loss')
    ax.set_xlabel('epoch')
    ax.set_ylabel('loss')
    ax.legend()
    plt.savefig("loss.png")
    plt.show()


In [91]:
class Accumulator:
    def __init__(self,n):
        self.data=[0.0]*n
    def add(self,*args):
        self.data=[x+y for x,y in zip(self.data,args)]
    def reset(self):
        self.data=[0.0]*len(self.data)
    def __getitem__(self,idx):
        return self.data[idx]
    def __str__(self):
        return str(self.data)

In [92]:
def translate(sentence):
    src_token=[token.text.lower() for token in spacy_en(sentence)]
    index=[src.vocab.stoi[token] for token in src_token]
    src_tensor=torch.tensor(index).unsqueeze(0)
    pad_mask=(src_tensor!=src_pad_idx).unsqueeze(1).unsqueeze(2)
    model.eval()
    with torch.no_grad():
        memory=model.encoder(src_tensor,src_mask=pad_mask)
    trg_indices=[trg.vocab.stoi['<sos>']]
    trg_tensor=torch.tensor(trg_indices).unsqueeze(0)
    look_ahead_mask=torch.tril(torch.ones(trg_tensor.size(1),trg_tensor.size(1))).unsqueeze(0)
    for i in range(20):
       
       with torch.no_grad():
           
           output=model.decoder(trg_tensor,memory,src_mask=pad_mask,trg_mask=look_ahead_mask)
       last_token=output[:,-1,:]
       pred_token=last_token.argmax(dim=-1)
       trg_indices.append(pred_token.item())
       trg_tensor=torch.cat((trg_tensor,pred_token.unsqueeze(0)),dim=1)

       if pred_token.item()==trg.vocab.stoi['<eos>']:
           break
    trans_tokens=[trg.vocab.itos[i] for i in trg_indices]
    trans_sentence=" ".join(trans_tokens)
    print("translate sentence:",trans_sentence)




In [93]:
def train1(model,train_iter,valid_iter,num_epoches):
    y_train_loss=[]
    y_valid_loss=[]
    epoch_num=[]
    optimizer=optim.Adam(model.parameters(),lr=0.0001,betas=(0.9,0.98),eps=1e-9)
    criterion=nn.CrossEntropyLoss(ignore_index=trg_pad_idx)
    # scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer,factor=0.1,patience=10)
    for epoch in range(num_epoches):
        epoch_num.append(epoch+1)
        metric=Accumulator(2)
        metric1=Accumulator(2)
        model.train()
        for i, batch in enumerate(train_iter):
            src=batch.src
            trg=batch.trg
            optimizer.zero_grad()
            output=model(src,trg[:,:-1])
            output_dim=output.shape[-1]
            output=output.contiguous().view(-1,output_dim)
            trg=trg[:,1:].contiguous().view(-1)
            loss=criterion(output,trg)#在实际过程中，交叉熵损失函数会自动进行softmax，并将其最后一个维度的最大值与目标标签相比较
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                metric.add(loss*src.shape[0],src.shape[0])
            train_loss=metric[0]/metric[1]
        y_train_loss.append(train_loss)
        for i,batch in enumerate(valid_iter):
            src=batch.src
            trg=batch.trg
            output=model(src,trg[:,:-1])
            output_dim=output.shape[-1]
            output=output.contiguous().view(-1,output_dim)
            trg=trg[:,1:].contiguous().view(-1)
            loss=criterion(output,trg)
            metric1.add(loss*src.shape[0],src.shape[0])
            valid_loss=metric1[0]/metric1[1]
        y_valid_loss.append(valid_loss)
       
        print(f'epoch:{epoch+1},train_loss:{train_loss:.3f},valid_loss:{valid_loss:.3f}')
    # scheduler.step()
    sentence='I am a student'
    translate(sentence)
    # np.savez('loss.npz',epoch=epoch_num,train_loss=y_train_loss,valid_loss=y_valid_loss)
    # vision(epoch_num,y_train_loss,y_valid_loss)
    return train_loss,valid_loss              



In [94]:
train1(model,train_iter,valid_iter,20)

epoch:1,train_loss:6.430,valid_loss:5.144
epoch:2,train_loss:4.783,valid_loss:4.415
epoch:3,train_loss:4.329,valid_loss:4.107
epoch:4,train_loss:4.089,valid_loss:3.916
epoch:5,train_loss:3.920,valid_loss:3.777
epoch:6,train_loss:3.784,valid_loss:3.636
epoch:7,train_loss:3.669,valid_loss:3.531
epoch:8,train_loss:3.571,valid_loss:3.440
epoch:9,train_loss:3.484,valid_loss:3.367
epoch:10,train_loss:3.406,valid_loss:3.294
epoch:11,train_loss:3.332,valid_loss:3.233
epoch:12,train_loss:3.267,valid_loss:3.168
epoch:13,train_loss:3.202,valid_loss:3.105
epoch:14,train_loss:3.142,valid_loss:3.049
epoch:15,train_loss:3.084,valid_loss:2.995
epoch:16,train_loss:3.029,valid_loss:2.967
epoch:17,train_loss:2.976,valid_loss:2.913
epoch:18,train_loss:2.929,valid_loss:2.868
epoch:19,train_loss:2.884,valid_loss:2.837
epoch:20,train_loss:2.843,valid_loss:2.790
translate sentence: <sos> eine gruppe von <unk> . <eos>


(tensor(2.8430), tensor(2.7900, grad_fn=<DivBackward0>))

In [95]:
src_token="I love machine learning"
src_token=[token.text.lower() for token in spacy_en(src_token)]
print(src_token)

['i', 'love', 'machine', 'learning']


In [96]:
index=[src.vocab.stoi[token] for token in src_token]
index

[956, 2169, 382, 1902]

In [97]:
# index1=[src.vocab.get(tok.lemma_) for tok in src_token]
# index1.append(src.vocab.stoi['<eos>'])

In [98]:
src_tensor=torch.tensor(index).unsqueeze(0)

In [99]:
pad_mask=(src_tensor!=src_pad_idx).unsqueeze(1).unsqueeze(2)

In [100]:
model.eval()
with torch.no_grad():
    memory=model.encoder(src_tensor,src_mask=pad_mask)
trg_indices=[trg.vocab.stoi['<sos>']]
trg_tensor=torch.tensor(trg_indices).unsqueeze(0)
look_ahead_mask=torch.tril(torch.ones(trg_tensor.size(1),trg_tensor.size(1))).unsqueeze(0)
for i in range(20):
    with torch.no_grad():
        output=model.decoder(trg_tensor,memory,src_mask=pad_mask,trg_mask=look_ahead_mask)
    last_token=output[:,-1,:]
    pred_token=last_token.argmax(dim=-1)
    trg_indices.append(pred_token.item())
    trg_tensor=torch.cat((trg_tensor,pred_token.unsqueeze(0)),dim=1)

    if pred_token.item()==trg.vocab.stoi['eos']:
        break
trans_tokens=[trg.vocab.itos[i] for i in trg_indices]
trans_sentence=" ".join(trans_tokens)
print("translate sentence:",trans_sentence)
        

translate sentence: <sos> <unk>


In [101]:
def translate(sentence):
    src_token=[token.text.lower() for token in spacy_en(sentence)]
    index=[src.vocab.stoi[token] for token in src_token]
    src_tensor=torch.tensor(index).unsqueeze(0)
    pad_mask=(src_tensor!=src_pad_idx).unsqueeze(1).unsqueeze(2)
    model.eval()
    with torch.no_grad():
        memory=model.encoder(src_tensor,src_mask=pad_mask)
    trg_indices=[trg.vocab.stoi['<sos>']]
    trg_tensor=torch.tensor(trg_indices).unsqueeze(0)
    look_ahead_mask=torch.tril(torch.ones(trg_tensor.size(1),trg_tensor.size(1))).unsqueeze(0)
    for i in range(20):
       
       with torch.no_grad():
           
           output=model.decoder(trg_tensor,memory,src_mask=pad_mask,trg_mask=look_ahead_mask)
       last_token=output[:,-1,:]
       pred_token=last_token.argmax(dim=-1)
       trg_indices.append(pred_token.item())
       trg_tensor=torch.cat((trg_tensor,pred_token.unsqueeze(0)),dim=1)

       if pred_token.item()==trg.vocab.stoi['eos']:
           break
    trans_tokens=[trg.vocab.itos[i] for i in trg_indices]
    trans_sentence=" ".join(trans_tokens)
    print("translate sentence:",trans_sentence)


